In [2]:
import mlflow
import pandas as pd
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Conectando com o banco SQLite local pra salvar meu histórico de testes
mlflow.set_tracking_uri("sqlite:///mlflow.db")
# Criando a "pasta" do projeto pra organizar as tentativas (runs)
mlflow.set_experiment("nyc-taxi-experiment")

# Puxando a base de dados dos táxis amarelos de NY (Janeiro de 2024)
df = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet")

# Dando aquela espiada rápida nas primeiras linhas pra ver se carregou certo
df.head()

# Criando o meu "gabarito" (coluna duration): subtraindo a hora de saída da hora de chegada e passando pra minutos
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Limpeza de dados vital! Tirando os outliers pra não estragar a IA (erro de 34 min nunca mais!)
df = df[(df.duration >= 1) & (df.duration <= 60)]

# Extrai uma amostra de 100 mil corridas para poupar a memória RAM do Codespaces
df = df.sample(n=100000, random_state=42)

# Definindo as minhas pistas (features)
categorical = ["PULocationID", "DOLocationID"] # locais
numerical = ["trip_distance"] # distância

# Forçando os locais a serem texto pro computador não achar que é matemática (que o bairro 10 vale mais que o 1)
df[categorical] = df[categorical].astype(str)

# Preparando as pistas pra jogar no tradutor
train_dicts = df[categorical + numerical].to_dict(orient="records")

# Iniciando o tradutor (DictVectorizer) que transforma texto em matriz
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

# Isolando a resposta certa (o gabarito real)
y_train = df["duration"].values

# ---------------------------------------------------------
# O MLflow "liga o gravador" para este teste específico
with mlflow.start_run():
    
    # Chamando o cérebro da operação (algoritmo de regressão linear)
    model = LinearRegression()
    
    # Treinando a IA com as pistas (X_train) e o gabarito (y_train)
    model.fit(X_train, y_train)
    
    # Fazendo a IA tentar adivinhar os dados pra ver como ela se sai
    y_pred = model.predict(X_train)
    
    # Calculando a margem de erro (RMSE) - quantos minutos ela tá errando na previsão
    rmse = mean_squared_error(y_train, y_pred) ** 0.5
    
    # O MLflow anota qual foi o erro médio desse teste pra eu ver lá no painel do navegador!
    mlflow.log_metric("rmse", rmse)